In [12]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_openai import ChatOpenAI



In [ ]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-5.2-2025-12-11"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


OpenAI API Key exists and begins sk-proj-


In [3]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 2 documents


In [4]:
# Divide into chunks using the RecursiveCharacterTextSplitter

#text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("###", "section")
    ]
)
chunks = []

for doc in documents:
    split_docs = header_splitter.split_text(doc.page_content)
    chunks.extend(split_docs)

print(f"Divided into {len(chunks)} chunks\n")

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content)
    print("\n")


Divided into 13 chunks

--- Chunk 1 ---
# UAE Golden Visa — Real Estate Investor Route  
## Overview
If you own a property (or group of properties) in the UAE, you may be eligible for a Golden Visa. Mortgaged properties may be accepted.  
## Eligibility


--- Chunk 2 ---
Real estate investors who own a property or a group of properties in the UAE.


--- Chunk 3 ---
- The total value of owned property/properties must be **at least AED 2,000,000**.
- If ownership is a **share in a jointly owned property**, the value of the applicant’s share must be **at least AED 2,000,000**.  
## Requirements


--- Chunk 4 ---
- Provide a **property status statement / ownership letter** issued by the relevant emirate’s land department (e.g., Dubai Land Department).
- Property value must be certified by the land department.
- A property valuation certificate may also be accepted if issued through **valuation offices licensed** by the Dubai Land Department.


--- Chunk 5 ---
- Purchasing a property with a

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large") -- more nuanced 

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 13 documents


In [6]:
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 13 vectors with 1,536 dimensions in the vector store


In [ ]:
DB_NAME = "vector_db"
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [16]:
retriever.invoke("what is the rules for golden visa?")

[Document(id='e7ce140d-69fe-4769-89b2-a7692fb3a34a', metadata={}, page_content='# UAE Golden Visa — Real Estate Investor Route  \n## Overview\nIf you own a property (or group of properties) in the UAE, you may be eligible for a Golden Visa. Mortgaged properties may be accepted.  \n## Eligibility'),
 Document(id='a5ff13e8-6c0c-4798-a1d0-65c3651b917f', metadata={'section': 'Lien requirement'}, page_content='- A **lien** may be required on the property to ensure continued ownership throughout the Golden Residency validity (per land department procedures).  \n## Processing time\n- Application completion time: **5 days**  \n## Key conditions to maintain the visa\n- One condition for maintaining the Golden Residence Permit is being able to **support yourself and your family** without requiring government support.\n- The authority may take measures to ensure individuals continue to meet the conditions throughout the visa validity.\n- During the residency period, the property may not be dispos

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing real estate broker who speciallises in off-plan properties in the UAE.
You are chatting with foreign investors who are interested in buying off-plan properties in the UAE.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content


answer_question("Who is Averi Lancaster?", [])


AIMessage(content="The UAE Golden Visa is a long-term residence visa that allows foreign investors, entrepreneurs, and talented individuals to live, work, and study in the UAE for extended periods, typically 5 or 10 years, with the possibility of renewal. Here are the key rules and eligibility criteria for obtaining a Golden Visa:\n\n1. **Eligibility Categories:**\n   - **Investors:** Those who invest a minimum amount in property or business.\n   - **Entrepreneurs:** Owners of successful startups or business owners with a certain turnover.\n   - **Talented Individuals:** Specialists in fields such as science, medicine, arts, and sports.\n   - **Researchers and Academics:** Distinguished researchers and university professors.\n   - **Students:** High-achieving students with outstanding academic records.\n\n2. **Investment Requirements:**\n   - **Property Investors:** Must invest at least AED 2 million in real estate (off-plan or ready properties). The property must be maintained for at 

In [8]:
# Prework


result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']

In [9]:
result = vectorstore._collection.get(include=["documents", "embeddings", "metadatas"])

i = 2 # pick which stored item to inspect
print("ID:", result["ids"][i])
print("METADATA:", result["metadatas"][i])
print("\nDOCUMENT:\n", result["documents"][i])

vec = result["embeddings"][i]
print("\nVECTOR LENGTH:", len(vec))
print("VECTOR (first 12 nums):", vec)


ID: b6a88efa-54aa-4d6e-844e-90fa8a4b8f30
METADATA: {'section': 'Proof of ownership / valuation'}

DOCUMENT:
 - Provide a **property status statement / ownership letter** issued by the relevant emirate’s land department (e.g., Dubai Land Department).
- Property value must be certified by the land department.
- A property valuation certificate may also be accepted if issued through **valuation offices licensed** by the Dubai Land Department.

VECTOR LENGTH: 1536
VECTOR (first 12 nums): [0.01777452 0.00552948 0.01849842 ... 0.02389983 0.01504598 0.02991376]


In [ ]:
import numpy as np

data = vectorstore._collection.get(include=["embeddings", "documents", "metadatas"])

vectors = np.array(data["embeddings"], dtype=np.float32)

# label each point by section (fallback to first line of text)
labels = []
for meta, doc in zip(data["metadatas"], data["documents"]):
    if isinstance(meta, dict) and meta.get("section"):
        labels.append(meta["section"])
    else:
        labels.append(doc.splitlines()[0][:30])


import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Reduce 1536D → 2D
n = vectors.shape[0]
perp = max(2, min(5, n - 1))  # safe for small datasets

tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=perp,
    init="pca",
    learning_rate="auto"
)

xy = tsne.fit_transform(vectors)

# Plot
plt.figure(figsize=(10, 7))
plt.scatter(xy[:, 0], xy[:, 1], s=40)

# Add labels
for i, label in enumerate(labels):
    plt.text(xy[i, 0], xy[i, 1], label[:15], fontsize=8)

plt.title("Your Vectorstore (2D Visualization)")
plt.xlabel("X")
plt.ylabel("Y")
plt.show()


In [ ]:
import plotly.graph_objects as go
from sklearn.manifold import TSNE
import numpy as np

# t-SNE (fix perplexity for small datasets)
tsne = TSNE(n_components=2, random_state=42, perplexity=3)
reduced_vectors = tsne.fit_transform(vectors)

# Create colors dynamically
colors = list(range(len(reduced_vectors)))  # simple color scale

# Optional hover labels
hover_text = labels  # from your earlier code

# Plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(
        size=8,
        color=colors,
        colorscale='Viridis',
        showscale=True,
        opacity=0.8
    ),
    text=hover_text,
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vector Store Visualization',
    xaxis_title='X',
    yaxis_title='Y',
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()
